# Ingestión del archivo `movie_languages.json`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo JSON usando `DataFrameReader` de Spark

In [0]:
movie_languages_schema = "movieId INT, languageId INT, languageRoleId INT"

movie_languages_df = (spark.read 
    .schema(movie_languages_schema)
    .option('multiLine', True)
    .json(f"{bronze_folder_path}/{v_file_date}/movie_language")
)
display(movie_languages_df)

movieId,languageId,languageRoleId
36593,24574,2
36597,24574,2
36643,24574,2
36647,24574,2
36648,24574,2
36657,24574,2
36658,24574,2
36668,24574,2
36669,24574,2
36670,24574,2


## 2. Eliminar las columnas no deseadas del DataFrame

In [0]:
movie_languages_dropped_df = movie_languages_df.drop("languageRoleId")

## 3. Cambiar el nombre de las columnas según lo requerido

In [0]:
movie_languages_renamed_df = (movie_languages_dropped_df
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("languageId", "language_id")
)

## 4. Agregar las columnas `ingestion_date` y `environmate` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

movie_languages_final_df = add_ingestion_date(movie_languages_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))


## 5. Escribir datos en el datalake en formato `Parquet`

In [0]:
merge_delta_lake( movie_languages_final_df, "movie_silver", "movies_languages", "tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.file_date = src.file_date", "file_date" )

In [0]:
%sql
SELECT * FROM movie_silver.movies_languages

movie_id,language_id,ingestion_date,enviroment
5,24574,2026-09-13T09:00:50.374771Z,developer
11,24574,2026-09-13T09:00:50.374771Z,developer
12,24574,2026-09-13T09:00:50.374771Z,developer
13,24574,2026-09-13T09:00:50.374771Z,developer
14,24574,2026-09-13T09:00:50.374771Z,developer
16,24574,2026-09-13T09:00:50.374771Z,developer
18,24574,2026-09-13T09:00:50.374771Z,developer
20,24574,2026-09-13T09:00:50.374771Z,developer
22,24574,2026-09-13T09:00:50.374771Z,developer
24,24574,2026-09-13T09:00:50.374771Z,developer
